# Phase 7: Feature Engineering

This notebook creates model-ready weekly forecasting features for Qty_Sold prediction.

## Objective
Build comprehensive feature set combining temporal, lag, rolling, business, external, and demand class features.

## Features Built
- **Date Features**: Year, Month, Quarter, ISO_Week, Week_Number, Month_Start, Month_End
- **Lag Features**: Lag_1, Lag_2, Lag_3, Lag_4, Lag_8, Lag_12 (per Product_ID)
- **Rolling Features**: Rolling_Mean_4, Rolling_Mean_8, Rolling_Std_4, Rolling_Max_4, Rolling_Min_4
- **Business Features**: Discount_Flag, Returns_Rate, Inventory_Ratio, Promo_Flag, Holiday_Flag
- **External Features**: Rainfall, Avg_Temperature, Season
- **Demand Class**: Smooth / Erratic / Intermittent / Lumpy

## Data Safety
- No target leakage (lags use shift before rolling)
- NaN handling from lags (first periods have NaN)
- Proper per-product grouping

## Inputs
- `data/processed/merged_dataset.csv`
- `data/processed/demand_classification.csv`

## Outputs
- `data/processed/feature_ready_dataset.csv`
- `reports/tables/feature_summary.csv`
- `reports/tables/null_summary_features.csv`
- `reports/charts/feature_correlation_heatmap.png`
- `reports/charts/demand_class_feature_mix.png`

## 1. Setup and Data Loading

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve project root whether executed from repo root or notebooks/ via nbconvert
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from features import build_features

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Imports complete')

Imports complete


In [2]:
# Load input datasets
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

merged_df = pd.read_csv(DATA_PROCESSED / 'merged_dataset.csv')
classification_df = pd.read_csv(DATA_PROCESSED / 'demand_classification.csv')

print(f'Merged dataset: {merged_df.shape}')
print(f'Classification: {classification_df.shape}')

Merged dataset: (2600, 21)
Classification: (25, 8)


## 2. Feature Engineering Pipeline

In [3]:
# Build all features using the feature engineering module
df_features = build_features(merged_df, classification_df)

print(f'Feature engineering complete')
print(f'Shape: {df_features.shape}')
print(f'\nColumns: {df_features.columns.tolist()}')

Feature engineering complete
Shape: (2600, 30)

Columns: ['Qty_Sold', 'Week_End_Date', 'Product_ID', 'Year', 'Month', 'Quarter', 'ISO_Week', 'Week_Number', 'Month_Start', 'Month_End', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_8', 'Lag_12', 'Rolling_Mean_4', 'Rolling_Mean_8', 'Rolling_Std_4', 'Rolling_Max_4', 'Rolling_Min_4', 'Discount_Flag', 'Returns_Rate', 'Inventory_Ratio', 'Promo_Flag', 'Holiday_Flag', 'Rainfall', 'Avg_Temperature', 'Season', 'Demand_Class']


## 3. Data Quality Checks

In [4]:
# Show sample rows
print('First 5 rows:')
print(df_features.head(5).to_string())

First 5 rows:
   Qty_Sold Week_End_Date Product_ID  Year  Month  Quarter  ISO_Week  Week_Number Month_Start                  Month_End  Lag_1  Lag_2  Lag_3  Lag_4  Lag_8  Lag_12  Rolling_Mean_4  Rolling_Mean_8  Rolling_Std_4  Rolling_Max_4  Rolling_Min_4  Discount_Flag  Returns_Rate  Inventory_Ratio  Promo_Flag  Holiday_Flag  Rainfall  Avg_Temperature  Season Demand_Class
0        69    2023-01-07      TE001  2023      1        1         1            1  2023-01-01 2023-01-31 23:59:59.999999    NaN    NaN    NaN    NaN    NaN     NaN             NaN             NaN            NaN            NaN            NaN              1      0.043478         7.304348           1             1     15.30            11.04  Winter       Smooth
1        73    2023-01-14      TE001  2023      1        1         2            2  2023-01-01 2023-01-31 23:59:59.999999   69.0    NaN    NaN    NaN    NaN     NaN       69.000000       69.000000            NaN           69.0           69.0              1      0.0

In [5]:
# Check for missing values
print('\nMissing values per column:')
missing = df_features.isnull().sum()
missing_pct = (missing / len(df_features)) * 100

missing_summary = pd.DataFrame({
    'Column': missing.index,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_pct.values
})
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_summary) > 0:
    print(missing_summary.to_string())
else:
    print('No missing values')


Missing values per column:
            Column  Missing_Count  Missing_Percentage
15          Lag_12            300           11.538462
14           Lag_8            200            7.692308
13           Lag_4            100            3.846154
12           Lag_3             75            2.884615
11           Lag_2             50            1.923077
18   Rolling_Std_4             50            1.923077
10           Lag_1             25            0.961538
16  Rolling_Mean_4             25            0.961538
17  Rolling_Mean_8             25            0.961538
19   Rolling_Max_4             25            0.961538
20   Rolling_Min_4             25            0.961538


In [6]:
# Data types and statistics
print('\nData types:')
print(df_features.dtypes)


Data types:
Qty_Sold                    int64
Week_End_Date      datetime64[us]
Product_ID                    str
Year                        int32
Month                       int32
Quarter                     int32
ISO_Week                   UInt32
Week_Number                 int64
Month_Start        datetime64[us]
Month_End          datetime64[us]
Lag_1                     float64
Lag_2                     float64
Lag_3                     float64
Lag_4                     float64
Lag_8                     float64
Lag_12                    float64
Rolling_Mean_4            float64
Rolling_Mean_8            float64
Rolling_Std_4             float64
Rolling_Max_4             float64
Rolling_Min_4             float64
Discount_Flag               int64
Returns_Rate              float64
Inventory_Ratio           float64
Promo_Flag                  int64
Holiday_Flag                int64
Rainfall                  float64
Avg_Temperature           float64
Season                        str
D

## 4. Save Outputs

In [7]:
# Save feature-ready dataset
output_path = DATA_PROCESSED / 'feature_ready_dataset.csv'
df_features.to_csv(output_path, index=False)
print(f'Feature-ready dataset saved: {output_path}')
print(f'  Shape: {df_features.shape}')
print(f'  Size: {output_path.stat().st_size / 1024:.1f} KB')

Feature-ready dataset saved: D:\Final Project\data\processed\feature_ready_dataset.csv
  Shape: (2600, 30)
  Size: 529.3 KB


## 5. Generate Reports

In [8]:
# Generate feature summary
REPORTS_TABLES = PROJECT_ROOT / 'reports' / 'tables'
REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

numeric_cols = df_features.select_dtypes(include=[np.number]).columns

feature_summary = pd.DataFrame({
    'Feature': df_features.columns,
    'Data_Type': df_features.dtypes.astype(str).values,
    'Missing_Count': df_features.isnull().sum().values,
    'Missing_Percentage': (df_features.isnull().sum() / len(df_features) * 100).values,
    'Min': [df_features[col].min() if col in numeric_cols else 'N/A' for col in df_features.columns],
    'Max': [df_features[col].max() if col in numeric_cols else 'N/A' for col in df_features.columns],
    'Mean': [df_features[col].mean() if col in numeric_cols else 'N/A' for col in df_features.columns],
})

feature_summary_path = REPORTS_TABLES / 'feature_summary.csv'
feature_summary.to_csv(feature_summary_path, index=False)
print(f'Feature summary saved: {feature_summary_path}')

Feature summary saved: D:\Final Project\reports\tables\feature_summary.csv


In [9]:
# Detailed null summary
null_summary = pd.DataFrame({
    'Column': df_features.columns,
    'Non_Null_Count': df_features.count().values,
    'Null_Count': df_features.isnull().sum().values,
    'Null_Percentage': (df_features.isnull().sum() / len(df_features) * 100).values,
    'Data_Type': df_features.dtypes.astype(str).values
})
null_summary = null_summary.sort_values('Null_Count', ascending=False)

null_summary_path = REPORTS_TABLES / 'null_summary_features.csv'
null_summary.to_csv(null_summary_path, index=False)
print(f'Null summary saved: {null_summary_path}')
print(f'\nColumns with missing values:')
print(null_summary[null_summary['Null_Count'] > 0].to_string())

Null summary saved: D:\Final Project\reports\tables\null_summary_features.csv

Columns with missing values:
            Column  Non_Null_Count  Null_Count  Null_Percentage Data_Type
15          Lag_12            2300         300        11.538462   float64
14           Lag_8            2400         200         7.692308   float64
13           Lag_4            2500         100         3.846154   float64
12           Lag_3            2525          75         2.884615   float64
11           Lag_2            2550          50         1.923077   float64
18   Rolling_Std_4            2550          50         1.923077   float64
16  Rolling_Mean_4            2575          25         0.961538   float64
20   Rolling_Min_4            2575          25         0.961538   float64
19   Rolling_Max_4            2575          25         0.961538   float64
17  Rolling_Mean_8            2575          25         0.961538   float64
10           Lag_1            2575          25         0.961538   float64


In [10]:
# Feature correlation heatmap (numeric features only)
numeric_features = df_features.select_dtypes(include=[np.number]).columns.tolist()
# Exclude Product_ID for correlation
numeric_features = [col for col in numeric_features if col != 'Product_ID']

if len(numeric_features) > 1:
    correlation_matrix = df_features[numeric_features].corr()
    
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, ax=ax, 
                cbar_kws={'label': 'Correlation'})
    ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    REPORTS_CHARTS = PROJECT_ROOT / 'reports' / 'charts'
    REPORTS_CHARTS.mkdir(parents=True, exist_ok=True)
    
    corr_path = REPORTS_CHARTS / 'feature_correlation_heatmap.png'
    fig.savefig(corr_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Correlation heatmap saved: {corr_path}')

Correlation heatmap saved: D:\Final Project\reports\charts\feature_correlation_heatmap.png


In [11]:
# Demand class feature mix visualization
if 'Demand_Class' in df_features.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Count plot
    demand_counts = df_features['Demand_Class'].value_counts()
    demand_counts.plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Demand Class Distribution (Records)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Demand Class')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Pie chart
    axes[1].pie(demand_counts, labels=demand_counts.index, autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Demand Class Distribution (Percentage)', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    
    demand_class_path = REPORTS_CHARTS / 'demand_class_feature_mix.png'
    fig.savefig(demand_class_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Demand class mix chart saved: {demand_class_path}')
    print(f'\nDemand Class Distribution:')
    print(demand_counts)

Demand class mix chart saved: D:\Final Project\reports\charts\demand_class_feature_mix.png

Demand Class Distribution:
Demand_Class
Smooth    2600
Name: count, dtype: int64


## 6. Summary

In [12]:
print('\n' + '='*70)
print('FEATURE ENGINEERING COMPLETE'.center(70))
print('='*70)
print(f'\nDataset Shape: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns')
print(f'\nFeature Categories:')
print('  - Date Features: Year, Month, Quarter, ISO_Week, Week_Number, Month_Start, Month_End')
print('  - Lag Features: Lag_1 through Lag_12')
print('  - Rolling Features: Mean, Std, Max, Min')
print('  - Business Features: Discount_Flag, Returns_Rate, Inventory_Ratio, Promo_Flag, Holiday_Flag')
print('  - External Features: Rainfall, Avg_Temperature, Season')
print('  - Demand Class: Smooth, Erratic, Intermittent, Lumpy')
print(f'\nOutputs:')
print(f'  - Main: data/processed/feature_ready_dataset.csv')
print(f'  - Tables: reports/tables/feature_summary.csv')
print(f'  - Tables: reports/tables/null_summary_features.csv')
print(f'  - Charts: reports/charts/feature_correlation_heatmap.png')
print(f'  - Charts: reports/charts/demand_class_feature_mix.png')
print('\nPhase 7 complete')


                     FEATURE ENGINEERING COMPLETE                     

Dataset Shape: 2,600 rows x 30 columns

Feature Categories:
  - Date Features: Year, Month, Quarter, ISO_Week, Week_Number, Month_Start, Month_End
  - Lag Features: Lag_1 through Lag_12
  - Rolling Features: Mean, Std, Max, Min
  - Business Features: Discount_Flag, Returns_Rate, Inventory_Ratio, Promo_Flag, Holiday_Flag
  - External Features: Rainfall, Avg_Temperature, Season
  - Demand Class: Smooth, Erratic, Intermittent, Lumpy

Outputs:
  - Main: data/processed/feature_ready_dataset.csv
  - Tables: reports/tables/feature_summary.csv
  - Tables: reports/tables/null_summary_features.csv
  - Charts: reports/charts/feature_correlation_heatmap.png
  - Charts: reports/charts/demand_class_feature_mix.png

Phase 7 complete
